# MiCA fine-tuned DistilBERT learning curves

This notebook measures how MiCA fine-tuned DistilBERT's training and validation accuracy change as the training set grows. The validation set stays fixed while the model is retrained on progressively larger training subsets.

Every point starts from the same pretrained checkpoint and uses three epochs. Early stopping is omitted so every point has the same training budget. The test split is not used.


In [ ]:
import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import DatasetDict, load_dataset
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, TaskType, get_peft_model


In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


## Load the training and validation data


In [ ]:
dataset = load_dataset("rasbt/human-vs-ai-50k")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "human": split["label"].count(0),
            "ai": split["label"].count(1),
            "total": len(split),
        }
        for name, split in {
            "train": train_dataset,
            "validation": validation_dataset,
        }.items()
    ]
).set_index("split")

split_summary


## Tokenize once

Tokenization is shared across all training runs. Each model keeps the same context length and readout construction used in its final training notebook.


In [ ]:
RANDOM_STATE = 17
NUM_TRAIN_EPOCHS = 3
TRAIN_FRACTIONS = np.asarray(
    [0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 1.00]
)

set_seed(RANDOM_STATE)

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 512


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


columns_to_remove = [
    column
    for column in train_dataset.column_names
    if column != "label"
]
curve_dataset = DatasetDict(
    {
        "train": train_dataset,
        "validation": validation_dataset,
    }
)
tokenized_dataset = curve_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=columns_to_remove,
    desc="Tokenizing",
)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

tokenized_dataset


## Construct nested training subsets

The subsets contain 1%, 2.5%, 5%, 10%, 25%, 50%, and 100% of the training data. A distribution-aware ordering keeps the joint label and source-collection distribution approximately stable. Each larger subset contains all samples from the preceding smaller subset.


In [ ]:
def make_distribution_aware_order(
    labels, source_collections, seed
):
    labels = np.asarray(labels, dtype=np.int64)
    source_collections = np.asarray(source_collections, dtype=str)
    strata = np.asarray(
        [
            f"{label}::{source}"
            for label, source in zip(labels, source_collections)
        ]
    )
    rng = np.random.default_rng(seed)
    groups = {}
    for stratum in sorted(np.unique(strata)):
        indices = np.flatnonzero(strata == stratum)
        rng.shuffle(indices)
        groups[stratum] = indices.tolist()

    total = labels.size
    proportions = {
        stratum: len(indices) / total
        for stratum, indices in groups.items()
    }
    selected = {stratum: 0 for stratum in groups}
    order = []
    for step in range(total):
        available = [
            stratum
            for stratum, indices in groups.items()
            if selected[stratum] < len(indices)
        ]
        stratum = max(
            available,
            key=lambda name: (
                proportions[name] * (step + 1) - selected[name],
                name,
            ),
        )
        order.append(groups[stratum][selected[stratum]])
        selected[stratum] += 1

    assert sorted(order) == list(range(total))
    return np.asarray(order, dtype=np.int64)


nested_order = make_distribution_aware_order(
    labels=train_dataset["label"],
    source_collections=train_dataset["source_collection"],
    seed=RANDOM_STATE,
)
train_sizes = np.rint(
    TRAIN_FRACTIONS * len(train_dataset)
).astype(np.int64)
train_sizes = np.unique(
    np.clip(train_sizes, 2, len(train_dataset))
)
train_sizes[-1] = len(train_dataset)

previous_indices = set()
for sample_count in train_sizes:
    current_indices = set(nested_order[:sample_count].tolist())
    assert previous_indices.issubset(current_indices)
    previous_indices = current_indices

subset_summary = pd.DataFrame(
    [
        {
            "training_samples": int(sample_count),
            "training_data_used": sample_count / len(train_dataset),
            "ai_share": np.mean(
                np.asarray(train_dataset["label"])[
                    nested_order[:sample_count]
                ]
            ),
        }
        for sample_count in train_sizes
    ]
)
subset_summary.style.format(
    {
        "training_data_used": "{:.1%}",
        "ai_share": "{:.1%}",
    }
)


## Fit one fresh model per training-set size

Each run reloads the original pretrained checkpoint, reconstructs any adapter, and resets the random seed. Training always lasts three epochs, so every example is seen the same number of times. The subset fractions sum to 1.935, so the complete curve requires almost twice the work of one full-data three-epoch run. Measurements are saved after every completed point.


In [ ]:
id2label = {0: "HUMAN", 1: "AI"}
label2id = {"HUMAN": 0, "AI": 1}


def build_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
    )
    mica_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=4,
        lora_alpha=8,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"],
        modules_to_save=["pre_classifier", "classifier"],
        bias="none",
        init_lora_weights="mica",
    )
    model = get_peft_model(base_model, mica_config)
    mica_a = [
        parameter
        for name, parameter in model.named_parameters()
        if "lora_A" in name
    ]
    mica_b = [
        parameter
        for name, parameter in model.named_parameters()
        if "lora_B" in name
    ]
    assert mica_a and mica_b
    assert all(parameter.requires_grad for parameter in mica_a)
    assert all(not parameter.requires_grad for parameter in mica_b)
    assert all(
        torch.count_nonzero(parameter).item() == 0
        for parameter in mica_a
    )
    return model


def find_project_dir():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "13_learning-curves").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the ai-detector-from-scratch project")


PROJECT_DIR = find_project_dir()
STAGE_DIR = PROJECT_DIR / "scripts" / "13_learning-curves"
RESULTS_DIR = STAGE_DIR / "results"
FIGURES_DIR = STAGE_DIR / "figures"
CHECKPOINT_ROOT = PROJECT_DIR / "checkpoints-distilbert-mica-learning-curves"
RESULTS_PATH = RESULTS_DIR / "distilbert-mica-learning-curve-results.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training epochs per point: {NUM_TRAIN_EPOCHS}")
print(f"Results file: {RESULTS_PATH}")


In [ ]:
results = []

for sample_count in train_sizes:
    sample_count = int(sample_count)
    train_indices = nested_order[:sample_count]
    train_subset = tokenized_dataset["train"].select(
        train_indices
    )

    set_seed(RANDOM_STATE)
    model = build_model()
    training_args = TrainingArguments(
        output_dir=str(
            CHECKPOINT_ROOT / f"samples-{sample_count}"
        ),
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        warmup_ratio=0.1,
        bf16=(
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        ),
        fp16=(
            torch.cuda.is_available()
            and not torch.cuda.is_bf16_supported()
        ),

        num_train_epochs=NUM_TRAIN_EPOCHS,
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_subset,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    print(f"\nTraining with {sample_count:,} samples")
    train_result = trainer.train()
    train_output = trainer.predict(train_subset)
    validation_output = trainer.predict(
        tokenized_dataset["validation"]
    )

    training_accuracy = accuracy_score(
        train_output.label_ids,
        np.argmax(train_output.predictions, axis=1),
    )
    validation_accuracy = accuracy_score(
        validation_output.label_ids,
        np.argmax(validation_output.predictions, axis=1),
    )
    results.append(
        {
            "training_samples": sample_count,
            "training_fraction": sample_count / len(train_dataset),
            "training_accuracy": training_accuracy,
            "validation_accuracy": validation_accuracy,
            "generalization_gap": (
                training_accuracy - validation_accuracy
            ),
            "training_runtime_seconds": (
                train_result.metrics["train_runtime"]
            ),
        }
    )
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
    print(
        f"Training accuracy: {training_accuracy:.2%}; "
        f"validation accuracy: {validation_accuracy:.2%}"
    )

    del trainer, model, train_subset
    del train_output, validation_output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

curve_results = pd.DataFrame(results)
curve_results


## Plot the learning curves


In [ ]:
x_percent = curve_results["training_fraction"].to_numpy() * 100
training_accuracy = curve_results["training_accuracy"].to_numpy()
validation_accuracy = curve_results["validation_accuracy"].to_numpy()

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(
    x_percent, training_accuracy,
    marker="o", linewidth=2, color="#1f77b4",
    label="Training accuracy",
)
ax.plot(
    x_percent, validation_accuracy,
    marker="o", linewidth=2, color="#666666",
    label="Validation accuracy",
)
ax.set_xscale("log")
ax.set_xticks(
    x_percent,
    [
        f"{fraction:.1f}%\n{samples:,}"
        for fraction, samples in zip(
            x_percent, curve_results["training_samples"]
        )
    ],
)
lower_limit = max(
    0.5,
    min(training_accuracy.min(), validation_accuracy.min()) - 0.03,
)
ax.set_ylim(lower_limit, 1.005)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Training data used")
ax.set_ylabel("Accuracy")
ax.set_title("MiCA fine-tuned DistilBERT learning curves", loc="left")
ax.grid(axis="y", color="#dddddd", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()

FIGURE_PATH = FIGURES_DIR / "distilbert-mica-learning-curves.svg"
fig.savefig(FIGURE_PATH, bbox_inches="tight")
plt.show()
print(f"Saved figure to {FIGURE_PATH}")


In [ ]:
curve_results.style.format(
    {
        "training_fraction": "{:.1%}",
        "training_accuracy": "{:.2%}",
        "validation_accuracy": "{:.2%}",
        "generalization_gap": "{:.2%}",
        "training_runtime_seconds": "{:,.1f}",
    }
)


## Interpreting the result

A large gap between training and validation accuracy suggests overfitting. If both curves remain low and close together, the model is more likely underfitting. If validation accuracy is still rising at the largest training-set sizes, collecting more training data may help. A flat validation curve suggests that additional samples from the same distribution may provide only small improvements.

These conclusions apply to the current data distribution and training configuration. A stronger robustness study could repeat every point with several random seeds and report the mean and standard deviation.
